In [0]:
%pip install paramiko tqdm

Looking in indexes: https://repo.eap.aon.com/artifactory/api/pypi/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/223.9 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.9/223.9 kB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/78.4 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/275.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 34.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/160.3 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/818.3 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.3/818.3 kB 65.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/216.5 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 38.1 MB/s eta 0:00:00

In [0]:
dbutils.library.restartPython()

In [0]:
#Define file-mover function
def move(src, dst_dir):
   dst = f"{dst_dir.rstrip('/')}/{src.split('/')[-1]}"
   dbutils.fs.mv(src, dst)
   return dst

In [0]:
import json
import boto3

#Get service account secrets for access to MFT folders

def get_aws_key_id_details(secret_name, session, region):
    client = session.client(service_name='secretsmanager', region_name=region)
    get_secret_value = client.get_secret_value(SecretId=secret_name)
    secrets_json = json.loads(get_secret_value['SecretString'])
    access_id = secrets_json['username']
    access_key = secrets_json['password']
    return access_id, access_key

session = boto3.session.Session()
region = 'us-east-1'
secret_name = 'prod/agrcnaanalytics/databricks/svc_dasp_agrcnaanalytics_dataeng'
sftp_username, sftp_password = get_aws_key_id_details(secret_name, session, region)

In [0]:
import paramiko
from tqdm import tqdm
import os
import stat
import posixpath

# SFTP server details
hostname = "web.mft4.aon.com"
port = 22

In [0]:
#Create SFTP connection to MFT folder
transport = paramiko.Transport((hostname, port))
transport.connect(username=sftp_username, password=sftp_password)
sftp = paramiko.SFTPClient.from_transport(transport)

In [0]:
#Define Databricks-local volume paths and MFT output folder path
remote_dir = "Commercial Risk/NA/DASP/Databricks/Triage/Outbound"
local_dir = "dbfs:/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/output/"
archive_dir = "dbfs:/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/archive/"

In [0]:
#Loops through files in Databricks output folder. For each file, uploads to MFT output folder and moves from Databricks output folder to Databricks archive folder.
output_files = dbutils.fs.ls(local_dir)
for file in output_files:
   local_file = file.path.replace("dbfs:/", "/")
   remote_file = posixpath.join(remote_dir, os.path.basename(file.path))
   sftp.put(local_file, remote_file)
   print(f"Uploaded to MFT: {remote_file}")
   archived_path = move(file.path, archive_dir)
   print(f"Archived file to: {archived_path}")

Uploaded to MFT: Commercial Risk/NA/DASP/Databricks/Triage/Outbound/Ace_Consultant_Triage_Draft.xlsx
Archived file to: dbfs:/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/archive/Ace_Consultant_Triage_Draft.xlsx
Uploaded to MFT: Commercial Risk/NA/DASP/Databricks/Triage/Outbound/Ace_DQ_Assessment.xlsx
Archived file to: dbfs:/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/archive/Ace_DQ_Assessment.xlsx
Uploaded to MFT: Commercial Risk/NA/DASP/Databricks/Triage/Outbound/Amrize_Consultant_Triage_Draft.xlsx
Archived file to: dbfs:/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/archive/Amrize_Consultant_Triage_Draft.xlsx
Uploaded to MFT: Commercial Risk/NA/DASP/Databricks/Triage/Outbound/Amrize_DQ_Assessment.xlsx
Archived file to: dbfs:/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/archive/Amrize_DQ_Assessment.xlsx
Uploaded to MFT: Commercial Risk/NA/DASP/Databricks/Triage/Outbound/Cemex_Consultant_Triage_Draft.xlsx
Archived file to: dbfs:/Volumes/a

In [0]:
#Close SFTP connection with MFT
sftp.close()
transport.close()